# Substep 1a - Training ActionFormer on EgoVLP 1-second features

This notebook produces one artefact: an ActionFormer checkpoint trained on our EgoVLP
features, saved to Drive. `extension_step1_actionformer.ipynb` consumes that checkpoint and
does everything the pipeline needs. Training lives here because it is a long GPU job that
should not re-run every time the pipeline runs.

**What is ours, and what is not.** The architecture, losses and decoding are the fork's.
Ours are: the EgoVLP 1-second feature representation the model is trained on, the config
that maps those features onto ActionFormer's temporal grid, the loader patch that lets it
read our `.npz` files, and everything downstream in Substeps 1-4.

Source: `github.com/rohithpeddi/actionformer_release`, the CaptainCook4D fork, pinned to a
commit so this notebook is reproducible.

In [1]:
# ==============================================================================
# 1. Environment, Drive and GPU
# ==============================================================================
import os
import sys
import json
import glob
import shutil
import subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/AML_Project"

# Training length. Section 12 derives which checkpoint file this produces rather than
# hardcoding a filename, so the promoted checkpoint, the evaluation and the export cannot
# drift apart the way they do when an epoch number is written down in three places.
NUM_EPOCHS = 28

REPO_DIR = "/content/actionformer_cc4d"
REPO_URL = "https://github.com/rohithpeddi/actionformer_release.git"
REPO_COMMIT = "0004e80367c21179ebfeae26e611d5ed69cbdc94"

FEATURE_DIR_DRIVE = os.path.join(DRIVE_BASE, "features/egovlp")
FEATURE_DIR = "/content/egovlp_1s"          # local copy; Drive I/O is far too slow
ANN_DIR = "/content/captaincook_annotations"
CKPT_ROOT = "/content/ckpt"
RUN_NAME = "cc4d_egovlp_1s"
DRIVE_CKPT_DIR = os.path.join(DRIVE_BASE, "step1/checkpoints", RUN_NAME)

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "no GPU visible - training will be unusably slow")
print(f"Training epochs requested: {NUM_EPOCHS}")

Mounted at /content/drive
NVIDIA L4, 23034 MiB
Training epochs requested: 28


In [2]:
# ==============================================================================
# 2. Stage the EgoVLP features on local disk
# ==============================================================================
# ActionFormer reads every feature file once per epoch. Served from the Drive mount that
# is minutes per epoch of pure I/O, so the features are copied to the Colab disk first.

os.makedirs(FEATURE_DIR, exist_ok=True)

existing = glob.glob(os.path.join(FEATURE_DIR, "*.npz"))
if not existing:
    src = sorted(glob.glob(os.path.join(FEATURE_DIR_DRIVE, "*.npz")))
    assert src, f"No .npz features found in {FEATURE_DIR_DRIVE}"
    print(f"Copying {len(src)} feature files to local disk...")
    for i, path in enumerate(src, 1):
        shutil.copy2(path, FEATURE_DIR)
        if i % 100 == 0:
            print(f"  {i}/{len(src)}")
    existing = glob.glob(os.path.join(FEATURE_DIR, "*.npz"))

print(f"Local feature files: {len(existing)}")
print(f"Example: {os.path.basename(existing[0])}")

Copying 384 feature files to local disk...
  100/384
  200/384
  300/384
Local feature files: 384
Example: 13_38_360p_224.npz


In [3]:
# ==============================================================================
# 3. Confirm the feature format and derive the input dimension
# ==============================================================================
import numpy as np

sample_path = sorted(glob.glob(os.path.join(FEATURE_DIR, "*.npz")))[0]
with np.load(sample_path) as data:
    keys = list(data.files)
    sample = data[keys[0]]

assert sample.ndim == 2, f"Expected T x C features, got {sample.shape}"
FEATURE_DIM = int(sample.shape[1])

print(f"File   : {os.path.basename(sample_path)}")
print(f"Keys   : {keys}")
print(f"Shape  : {sample.shape}  (T rows x C channels)")
print(f"FEATURE_DIM = {FEATURE_DIM}")
assert FEATURE_DIM == 256, (
    f"Expected 256-d EgoVLP features, found {FEATURE_DIM}. This notebook's config, and "
    "every downstream substep, assumes the EgoVLP representation.")

File   : 10_16_360p_224.npz
Keys   : ['video_features']
Shape  : (973, 256)  (T rows x C channels)
FEATURE_DIM = 256


In [4]:
# ==============================================================================
# 4. Clone the CaptainCook4D ActionFormer fork at a pinned commit
# ==============================================================================
# rohithpeddi/actionformer_release is the fork CaptainCook4D used for its own
# localization baseline: it carries the `error_dataset` loader and the `error_*.yaml`
# configs that the upstream ActionFormer repository does not have. Pinning the commit
# means this notebook does not silently change behaviour if the fork moves.

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--quiet"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REPO_COMMIT], check=True)

head = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
assert head == REPO_COMMIT, f"Checked out {head}, expected {REPO_COMMIT}"
print(f"actionformer_release @ {head}")
print("configs available:", sorted(os.path.basename(p)
                                   for p in glob.glob(os.path.join(REPO_DIR, "configs", "error_*.yaml"))))

actionformer_release @ 0004e80367c21179ebfeae26e611d5ed69cbdc94
configs available: ['error_omnivore.yaml']


In [5]:
# ==============================================================================
# 5. Dependencies and the 1D NMS extension
# ==============================================================================
# ActionFormer's Soft-NMS is a C++ extension that has to be compiled against the PyTorch
# in this runtime; a prebuilt .so from another environment will not import. `nms.py` does
# a top-level `import nms_1d_cpu`, so libs/utils itself must be importable, not just the
# repository root.

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tensorboard", "einops", "pyyaml", "pandas", "h5py", "joblib"], check=True)

UTILS_DIR = os.path.join(REPO_DIR, "libs", "utils")

if not glob.glob(os.path.join(UTILS_DIR, "nms_1d_cpu*.so")):
    print("Compiling the 1D NMS extension...")
    subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"],
                   cwd=UTILS_DIR, check=True)

built = glob.glob(os.path.join(UTILS_DIR, "nms_1d_cpu*.so"))
assert built, "NMS build reported success but produced no nms_1d_cpu*.so"
print("Compiled:", os.path.basename(built[0]))

for path in (UTILS_DIR, REPO_DIR):
    if path in sys.path:
        sys.path.remove(path)
sys.path.insert(0, UTILS_DIR)
sys.path.insert(0, REPO_DIR)

import torch                                         # Load PyTorch libraries first
import nms_1d_cpu                                    # noqa: F401
from libs.core import load_config                    # noqa: F401
from libs.datasets import make_dataset               # noqa: F401
from libs.modeling import make_meta_arch             # noqa: F401
print("ActionFormer imports resolve")

Compiling the 1D NMS extension...
Compiled: nms_1d_cpu.cpython-313-x86_64-linux-gnu.so
ActionFormer imports resolve


In [6]:
# ==============================================================================
# 6. Official CaptainCook4D localization annotations and split
# ==============================================================================
# These are the annotations CaptainCook4D published for its own ActionFormer baseline:
# one entry per recording with its step segments, class ids, and the subset it belongs to.
# Using them rather than a split of our own means our train/val/test boundary is the same
# one the published numbers were computed against.

import urllib.request

os.makedirs(ANN_DIR, exist_ok=True)
ANN_FILE = os.path.join(ANN_DIR, "recordings.json")
ANN_URL = ("https://raw.githubusercontent.com/CaptainCook4D/multi_step_localization/"
           "main/captaincook_actionformer_annotations/combined/recordings.json")

if not os.path.isfile(ANN_FILE):
    urllib.request.urlretrieve(ANN_URL, ANN_FILE)

with open(ANN_FILE) as f:
    db = json.load(f)["database"]

subsets = {}
label_ids = set()
for vid, item in db.items():
    subsets[item["subset"]] = subsets.get(item["subset"], 0) + 1
    for a in item.get("annotations", []):
        label_ids.add(int(a["label_id"]))

NUM_CLASSES = max(label_ids) + 1     # label ids are 1-based; index 0 stays unused

print(f"Recordings in the annotation database: {len(db)}")
print(f"Subsets: {subsets}")
print(f"Distinct step classes: {len(label_ids)}  (label_id {min(label_ids)}..{max(label_ids)})")
print(f"NUM_CLASSES passed to the model: {NUM_CLASSES}")

Recordings in the annotation database: 384
Subsets: {'Training': 213, 'Validation': 62, 'Test': 109}
Distinct step classes: 350  (label_id 1..352)
NUM_CLASSES passed to the model: 353


## Making the fork read our features

The fork's `error_dataset.py` expects one feature file per recording named exactly
`<prefix><recording_id><ext>`, and expects the array inside to be under a known key. Our
EgoVLP extraction writes files like `13_38_360p_224.npz` - the recording id followed by a
suffix recording how the features were produced - so the fork's exact-name lookup misses
every file.

The cell below rewrites that one lookup: try the exact name first, then fall back to a
glob on `<recording_id>_*`, and read whichever array key the archive actually contains. It
refuses to guess when a glob matches more than one file, because silently training on the
wrong feature variant is the failure this whole project has already been bitten by once.

Nothing else in the fork is modified. The patch is idempotent and the result is
syntax-checked, so re-running this cell is safe.

In [7]:
# ==============================================================================
# 7. Patch the fork's feature loader for our .npz naming
# ==============================================================================
import ast
import textwrap

loader_path = Path(REPO_DIR) / "libs" / "datasets" / "error_dataset.py"
original = loader_path.read_text()

PATCH_MARK = "# --- EgoVLP filename/key compatibility (patched by substep 1a) ---"

if PATCH_MARK in original:
    print("Loader already patched; nothing to do.")
else:
    start_marker = "\t\t# load features\n"
    end_marker = "\n\t\t# deal with downsampling"
    start, end = original.find(start_marker), original.find(end_marker)
    assert start != -1 and end != -1, (
        "Could not locate the feature-loading block in error_dataset.py. The pinned "
        "commit should contain it; check REPO_COMMIT.")

    replacement = textwrap.indent(textwrap.dedent(f'''
        {PATCH_MARK}
        base_id = video_item['id']
        filename = os.path.join(
            self.feat_folder, self.file_prefix + base_id + self.file_ext)

        if not os.path.isfile(filename):
            pattern = os.path.join(
                self.feat_folder, self.file_prefix + base_id + "_*" + self.file_ext)
            matches = sorted(glob.glob(pattern))
            if len(matches) == 0:
                raise FileNotFoundError(
                    f"No feature file for {{base_id}} (looked for {{filename}} then {{pattern}})")
            if len(matches) > 1:
                raise RuntimeError(
                    f"Ambiguous feature files for {{base_id}}: {{matches}}. Refusing to "
                    f"guess which feature variant to train on.")
            filename = matches[0]

        with np.load(filename) as _npz:
            if "feats" in _npz.files:
                feats = _npz["feats"].astype(np.float32)
            elif len(_npz.files) == 1:
                feats = _npz[_npz.files[0]].astype(np.float32)
            else:
                raise KeyError(
                    f"Cannot identify the feature array in {{filename}}; keys={{_npz.files}}")
    ''').strip('\n'), '\t\t')

    patched = original[:start] + "\t\t# load features\n" + replacement + original[end:]
    if "import glob" not in patched:
        patched = patched.replace("import os\n", "import os\nimport glob\n", 1)

    ast.parse(patched)          # fail here rather than inside a dataloader worker
    loader_path.write_text(patched)
    print(f"Patched {loader_path}")

assert PATCH_MARK in loader_path.read_text()
ast.parse(loader_path.read_text())
print("Loader patch verified (parses, marker present).")

Patched /content/actionformer_cc4d/libs/datasets/error_dataset.py
Loader patch verified (parses, marker present).


In [8]:
# ==============================================================================
# 8. Cross-check annotation ids against the feature files
# ==============================================================================
# The patch above resolves ids to files by globbing. This confirms that resolution is
# total and unambiguous BEFORE a training run discovers it at epoch 3.

feature_files = glob.glob(os.path.join(FEATURE_DIR, "*.npz"))
missing, ambiguous = [], []

for rec_id in db:
    exact = os.path.join(FEATURE_DIR, rec_id + ".npz")
    if os.path.isfile(exact):
        continue
    matches = sorted(glob.glob(os.path.join(FEATURE_DIR, rec_id + "_*.npz")))
    if not matches:
        missing.append(rec_id)
    elif len(matches) > 1:
        ambiguous.append((rec_id, [os.path.basename(m) for m in matches]))

print(f"Annotated recordings : {len(db)}")
print(f"Feature files        : {len(feature_files)}")
print(f"Unresolvable ids     : {len(missing)}")
print(f"Ambiguous ids        : {len(ambiguous)}")
if missing:
    print("  missing:", missing[:10])
if ambiguous:
    print("  ambiguous:", ambiguous[:5])

assert not missing and not ambiguous, (
    "Every annotated recording must map to exactly one feature file before training.")
print("\nEvery annotated recording resolves to exactly one feature file.")

Annotated recordings : 384
Feature files        : 384
Unresolvable ids     : 0
Ambiguous ids        : 0

Every annotated recording resolves to exactly one feature file.


In [9]:
# ==============================================================================
# 9. The EgoVLP 1-second config
# ==============================================================================
# Start from the fork's own CaptainCook4D localization config and override only what the
# change of feature representation requires. Everything not listed here - the backbone
# shape, the FPN, the regression ranges, the loss, the Soft-NMS test config - is left at
# the values CaptainCook4D used.
#
# The temporal fields are the ones that matter and the ones easiest to get wrong:
#
#   feat_stride / default_fps = 30 / 30 = 1.0 second per feature row
#
# The published baseline used Omnivore features at 4 s per row. Ours are EgoVLP at 1 s per
# row, so feat_stride and num_frames both become 30 against a 30 fps source convention.
# ActionFormer converts its temporal grid back to seconds with this ratio, so getting it
# wrong does not error - it silently returns segments in the wrong time base.

import yaml

with open(os.path.join(REPO_DIR, "configs", "error_omnivore.yaml")) as f:
    cfg = yaml.safe_load(f)

cfg["dataset_name"] = "error"
cfg["train_split"] = ["training"]
cfg["val_split"] = ["validation"]
cfg["output_folder"] = CKPT_ROOT + "/"

d = cfg["dataset"]
d["json_file"] = ANN_FILE
d["feat_folder"] = FEATURE_DIR
d["file_prefix"] = None
d["file_ext"] = ".npz"
d["num_classes"] = int(NUM_CLASSES)
d["input_dim"] = FEATURE_DIM
d["feat_stride"] = 30
d["num_frames"] = 30
d["default_fps"] = 30
d["max_seq_len"] = 1024

cfg["model"]["input_dim"] = FEATURE_DIM
cfg["opt"]["epochs"] = NUM_EPOCHS
cfg["loader"]["batch_size"] = 2
cfg["loader"]["num_workers"] = 2

seconds_per_row = d["feat_stride"] / d["default_fps"]
assert abs(seconds_per_row - 1.0) < 1e-6, (
    f"Config gives {seconds_per_row}s per feature row; the EgoVLP features are 1s per row.")

CONFIG_PATH = os.path.join(REPO_DIR, "configs", "captaincook_egovlp_1s.yaml")
with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Wrote {CONFIG_PATH}")
print(f"  input_dim   : {FEATURE_DIM}")
print(f"  num_classes : {NUM_CLASSES}")
print(f"  time base   : {seconds_per_row:.4f} s per feature row")
print(f"  epochs      : {NUM_EPOCHS}")

Wrote /content/actionformer_cc4d/configs/captaincook_egovlp_1s.yaml
  input_dim   : 256
  num_classes : 353
  time base   : 1.0000 s per feature row
  epochs      : 28


In [10]:
# ==============================================================================
# 10. Dataset sanity check before committing to a long run
# ==============================================================================
# Builds the dataset through the fork's own factory and inspects one batch. This catches
# the time-base and dimension errors that would otherwise only show up as a model that
# trains to a plausible-looking loss and predicts segments in the wrong units.

from libs.core import load_config as _load_config
from libs.datasets import make_dataset as _make_dataset
import libs.datasets.error_dataset

_cfg = _load_config(CONFIG_PATH)
_train = _make_dataset(_cfg['dataset_name'], True, _cfg['train_split'], **_cfg['dataset'])

print(f"Training recordings: {len(_train)}")
_item = _train[0]
print(f"Example id      : {_item['video_id']}")
print(f"Feature tensor  : {tuple(_item['feats'].shape)}  (C x T)")
print(f"fps             : {_item['fps']}")
print(f"feat_stride     : {_item['feat_stride']}")
print(f"segments (s)    : {_item['segments'][:3].tolist()}")
print(f"labels          : {_item['labels'][:3].tolist()}")

assert _item['feats'].shape[0] == FEATURE_DIM, (
    f"Loader produced {_item['feats'].shape[0]} channels, expected {FEATURE_DIM}")

_duration_rows = _item['feats'].shape[1]
_max_end = float(_item['segments'].max()) if len(_item['segments']) else 0.0
print(f"\nRows: {_duration_rows}   last annotated second: {_max_end:.1f}")
assert _max_end <= _duration_rows * 1.5, (
    "Annotated times run far past the feature rows - the time base is inconsistent.")
print("Dataset, dimensions and time base look consistent.")

Training recordings: 213
Example id      : 1_19
Feature tensor  : (256, 646)  (C x T)
fps             : 30
feat_stride     : 30
segments (s)    : [[-0.5, 14.172550201416016], [18.68960189819336, 60.714996337890625], [80.18395233154297, 134.56044006347656]]
labels          : [3, 1, 4]

Rows: 646   last annotated second: 635.0
Dataset, dimensions and time base look consistent.


## Train

One run, `NUM_EPOCHS` epochs, on the official training split. The fork writes a checkpoint
every `--ckpt-freq` epochs into `output_folder`, and the console log is kept so the loss
curve can be read back afterwards.

Wall-clock depends on the GPU Colab assigns: about ten minutes on an L4, longer on a T4.
Nothing below depends on the notebook staying connected beyond this cell - the next
section promotes a checkpoint to Drive, and the pipeline notebook reads it from there.

In [11]:
# ==============================================================================
# 11. Run training
# ==============================================================================
import time
import os

TRAIN_LOG = "/content/actionformer_train.log"
train_start = time.time()

cmd = [sys.executable, "train.py", CONFIG_PATH,
       "--output", RUN_NAME, "--ckpt-freq", "4", "--print-freq", "20"]
print("Running:", " ".join(cmd), "\n")

# Provide the compiled NMS extension to the subprocess
train_env = os.environ.copy()
train_env["PYTHONPATH"] = f"{UTILS_DIR}:{REPO_DIR}:" + train_env.get("PYTHONPATH", "")

init_path = os.path.join(REPO_DIR, "libs", "datasets", "__init__.py")
if "error_dataset" not in open(init_path).read():
    with open(init_path, "a") as f:
        f.write("\nfrom . import error_dataset\n")

with open(TRAIN_LOG, "w") as log:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=train_env)
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
    returncode = proc.wait()

assert returncode == 0, f"Training exited {returncode}; see {TRAIN_LOG}"
print(f"\nTraining finished in {(time.time() - train_start) / 60:.1f} minutes.")

Running: /usr/bin/python3 train.py /content/actionformer_cc4d/configs/captaincook_egovlp_1s.yaml --output cc4d_egovlp_1s --ckpt-freq 4 --print-freq 20 

2026-08-30 14:16:43.874457: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-30 14:16:43.942881: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
{'dataset': {'crop_ratio': [0.9, 1.0],
             'default_fps': 30,
             'downsample_rate': 1,
             'feat_folder': '/content/egovlp_1s',
             'feat_stride': 30,
             'file_ext': '.npz',


In [12]:
# ==============================================================================
# 12. Promote one checkpoint to Drive
# ==============================================================================
# ActionFormer trains for `opt.epochs + opt.warmup_epochs` epochs and names checkpoints by
# their zero-based index, so a config asking for 28 epochs with 5 warmup epochs produces a
# final `epoch_032.pth.tar`. Deriving that index from the config rather than hardcoding a
# filename is what stops the evaluation, the exported predictions and the reported metrics
# from silently referring to three different checkpoints.

from libs.core import load_config as _resolve_config

resolved = _resolve_config(CONFIG_PATH)
warmup = int(resolved["opt"].get("warmup_epochs", 5))
final_epoch_index = int(resolved["opt"]["epochs"]) + warmup - 1

EXP_DIR = os.path.join(CKPT_ROOT, f"captaincook_egovlp_1s_{RUN_NAME}")
available = sorted(glob.glob(os.path.join(EXP_DIR, "epoch_*.pth.tar")))
assert available, f"No checkpoints found in {EXP_DIR}"

SELECTED_CKPT = os.path.join(EXP_DIR, f"epoch_{final_epoch_index:03d}.pth.tar")
assert os.path.isfile(SELECTED_CKPT), (
    f"Expected the final checkpoint {os.path.basename(SELECTED_CKPT)} "
    f"(epochs={resolved['opt']['epochs']} + warmup={warmup}), but the run produced:\n  "
    + "\n  ".join(os.path.basename(p) for p in available))

print(f"Checkpoints written : {[os.path.basename(p) for p in available]}")
print(f"Selected            : {os.path.basename(SELECTED_CKPT)}  "
      f"({os.path.getsize(SELECTED_CKPT) / 1e6:.0f} MB)")

os.makedirs(os.path.dirname(DRIVE_CKPT_DIR), exist_ok=True)
if os.path.isdir(DRIVE_CKPT_DIR):
    shutil.rmtree(DRIVE_CKPT_DIR)
os.makedirs(DRIVE_CKPT_DIR)

# The pipeline notebook needs exactly three things: the weights, the resolved config that
# describes the architecture and time base, and a record of which epoch this was.
shutil.copy2(SELECTED_CKPT, os.path.join(DRIVE_CKPT_DIR, "checkpoint.pth.tar"))
shutil.copy2(CONFIG_PATH, os.path.join(DRIVE_CKPT_DIR, "captaincook_egovlp_1s.yaml"))
shutil.copy2(TRAIN_LOG, os.path.join(DRIVE_CKPT_DIR, "train_console.log"))
for extra in glob.glob(os.path.join(EXP_DIR, "config.txt")):
    shutil.copy2(extra, DRIVE_CKPT_DIR)

with open(os.path.join(DRIVE_CKPT_DIR, "provenance.json"), "w") as f:
    json.dump({
        "repo": REPO_URL,
        "commit": REPO_COMMIT,
        "source_checkpoint": os.path.basename(SELECTED_CKPT),
        "epoch_index": final_epoch_index,
        "config_epochs": int(resolved["opt"]["epochs"]),
        "warmup_epochs": warmup,
        "feature_dim": FEATURE_DIM,
        "num_classes": int(NUM_CLASSES),
        "seconds_per_feature_row": 1.0,
    }, f, indent=2)

print(f"\nPromoted to {DRIVE_CKPT_DIR}")
print("  checkpoint.pth.tar, captaincook_egovlp_1s.yaml, provenance.json, train_console.log")

Checkpoints written : ['epoch_004.pth.tar', 'epoch_008.pth.tar', 'epoch_012.pth.tar', 'epoch_016.pth.tar', 'epoch_020.pth.tar', 'epoch_024.pth.tar', 'epoch_028.pth.tar', 'epoch_032.pth.tar', 'epoch_033.pth.tar']
Selected            : epoch_032.pth.tar  (433 MB)

Promoted to /content/drive/MyDrive/AML_Project/step1/checkpoints/cc4d_egovlp_1s
  checkpoint.pth.tar, captaincook_egovlp_1s.yaml, provenance.json, train_console.log


## Evaluate the promoted checkpoint

This runs the fork's own `eval.py` on the **validation** split, using the checkpoint that
was just promoted - the same file the pipeline notebook will load. The mAP figures below
are therefore the ones that describe the exported predictions, not a different epoch's.

These are the fork's multi-class localization metrics over 353 step classes. They are not
the numbers Substep 1 reports downstream: the pipeline is class-agnostic and cares only
about *where* the step boundaries are, so `extension_step1_actionformer.ipynb` recomputes
class-agnostic precision/recall/F1 at a range of tIoU thresholds on the held-out test
split. Read these as a health check on training, and those as the localization result.

In [17]:
# ==============================================================================
# 13. Validation evaluation, on the promoted checkpoint
# ==============================================================================
EVAL_LOG = "/content/actionformer_eval.log"
EVAL_CKPT = os.path.join(DRIVE_CKPT_DIR, "checkpoint.pth.tar")

# Patch eval.py for PyTorch compatibility (map_location accepts the device string directly)
eval_script_path = os.path.join(REPO_DIR, "eval.py")
with open(eval_script_path, "r") as f:
    eval_code = f.read()
eval_code = eval_code.replace("lambda storage, loc: storage.cuda(cfg['devices'][0])", "cfg['devices'][0]")
with open(eval_script_path, "w") as f:
    f.write(eval_code)

# Patch the deprecated np.float / np.bool / np.int from older numpy versions across the repository
import re
for root, _, files in os.walk(REPO_DIR):
    for file in files:
        if file.endswith(".py"):
            py_path = os.path.join(root, file)
            with open(py_path, "r") as f:
                code = f.read()
            # Negative lookahead to ensure we don't accidentally replace np.float32 or np.int64
            new_code = re.sub(r'np\.float(?!\w)', 'float', code)
            new_code = re.sub(r'np\.bool(?!\w)', 'bool', new_code)
            new_code = re.sub(r'np\.int(?!\w)', 'int', new_code)
            if new_code != code:
                with open(py_path, "w") as f:
                    f.write(new_code)

cmd = [sys.executable, "eval.py", CONFIG_PATH, EVAL_CKPT]
print("Evaluating:", EVAL_CKPT, "\n")

eval_env = os.environ.copy()
eval_env["PYTHONPATH"] = f"{UTILS_DIR}:{REPO_DIR}:" + eval_env.get("PYTHONPATH", "")

with open(EVAL_LOG, "w") as log:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=eval_env)
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
    returncode = proc.wait()

assert returncode == 0, f"Evaluation exited {returncode}; see {EVAL_LOG}"
shutil.copy2(EVAL_LOG, os.path.join(DRIVE_CKPT_DIR, "eval_console.log"))

log_text = open(EVAL_LOG, errors="ignore").read()
metric_lines = [l for l in log_text.splitlines() if "mAP" in l or "Average" in l]

print("\n" + "=" * 78)
print("VALIDATION LOCALIZATION (fork's multi-class metrics, 353 classes)")
print("=" * 78)
for line in metric_lines:
    print(line)

assert metric_lines, "Evaluation produced no metric lines - check eval_console.log"
print(f"\nCheckpoint ready for the pipeline notebook:\n  {EVAL_CKPT}")

Evaluating: /content/drive/MyDrive/AML_Project/step1/checkpoints/cc4d_egovlp_1s/checkpoint.pth.tar 

{'dataset': {'crop_ratio': [0.9, 1.0],
             'default_fps': 30,
             'downsample_rate': 1,
             'feat_folder': '/content/egovlp_1s',
             'feat_stride': 30,
             'file_ext': '.npz',
             'file_prefix': None,
             'force_upsampling': False,
             'input_dim': 256,
             'json_file': '/content/captaincook_annotations/recordings.json',
             'max_seq_len': 1024,
             'num_classes': 353,
             'num_frames': 30,
             'trunc_thresh': 0.3},
 'dataset_name': 'error',
 'devices': ['cuda:0'],
 'init_rand_seed': 1234567891,
 'loader': {'batch_size': 2, 'num_workers': 2},
 'model': {'backbone_arch': (2, 2, 5),
           'backbone_type': 'convTransformer',
           'embd_dim': 512,
           'embd_kernel_size': 3,
           'embd_with_ln': True,
           'fpn_dim': 512,
           'fpn_start_lev